# 04D1G_v7 — Archived LANDFIRE Acquisition & Validation

**Build ID: `04D1G_V7_LANDFIRE_ARCHIVE_ACQUISITION_VALIDATION`**

04D1F V2/V3 resolved the archived source inventory:

- LF2019 Limited (`210`) → one unique CONUS archive per required product;
- LF2020 (`220`) → one unique ScienceBase CONUS archive per required product.

Required products:

- FBFM40
- CC
- CH
- CBH
- CBD

This notebook performs the first **large-data acquisition step**.

## Actions

1. load the reviewed LF2019L and LF2020 selections from 04D1F;
2. deduplicate and freeze exactly 10 archive URLs;
3. check disk capacity before downloading;
4. download each archive with `.part` resume support and retry;
5. compare downloaded byte count against source metadata;
6. compute a local SHA-256 for every archive;
7. validate ZIP central directory and optional full CRC;
8. inventory all ZIP members;
9. extract only GeoTIFF/XML/CSV metadata products;
10. validate GeoTIFF CRS/grid/dtype/NoData;
11. check within-vintage cross-product raster geometry;
12. perform small distributed raw-value spot checks;
13. write a machine-readable provenance manifest for later blockwise production.

The frozen 04D1C V5 feature architecture is unchanged.

## 0. Configuration

In [ ]:
import os  # Configurable project root for the public release.
NOTEBOOK_BUILD_ID = "04D1G_V7_LANDFIRE_ARCHIVE_ACQUISITION_VALIDATION"

from pathlib import Path
import sys
import subprocess
import importlib.util
import shutil

PROJECT_ROOT = Path(
    os.environ.get("FIRELINE_PROJECT_ROOT", ".")
)

V2_INVENTORY_ROOT = (
    PROJECT_ROOT / "outputs" /
    "landfire_archive_inventory_v2"
)

V3_INVENTORY_ROOT = (
    PROJECT_ROOT / "outputs" /
    "landfire_archive_inventory_v3"
)

LF2020_SELECTION_PATH = (
    V2_INVENTORY_ROOT /
    "LANDFIRE_ARCHIVE_RECOMMENDED_SELECTION_V2.csv"
)

LF2019L_PLAN_PATH = (
    V3_INVENTORY_ROOT /
    "LF2019L_ACQUISITION_SET_PROPOSAL_V3.csv"
)

OUTPUT_ROOT = (
    PROJECT_ROOT / "outputs" /
    "landfire_archive_acquisition_v7"
)

ARCHIVE_ROOT = (
    PROJECT_ROOT / "data" / "raw" /
    "landfire" /
    "archive_acquisition" /
    "archives"
)

EXTRACT_ROOT = (
    PROJECT_ROOT / "data" / "raw" /
    "landfire" /
    "archive_acquisition" /
    "extracted"
)

for p in [
    LF2020_SELECTION_PATH,
    LF2019L_PLAN_PATH,
]:
    assert p.exists(), p

for p in [
    OUTPUT_ROOT,
    ARCHIVE_ROOT,
    EXTRACT_ROOT,
]:
    p.mkdir(
        parents=True,
        exist_ok=True,
    )

REQUIRED_PRODUCTS = [
    "FBFM40",
    "CC",
    "CH",
    "CBH",
    "CBD",
]

# This notebook is intended to perform the acquisition.
DOWNLOAD_ARCHIVES = True

# Resume `.part` files after interruption.
RESUME_PARTIAL_DOWNLOADS = True

# Full ZIP CRC validation reads the complete local compressed archive after
# download. Keep True for provenance-quality acquisition.
RUN_FULL_ZIP_CRC_TEST = True

REQUEST_TIMEOUT_S = 180
DOWNLOAD_CHUNK_BYTES = 8 * 1024 * 1024

# V2 robust segmented downloader.
RANGE_SEGMENT_BYTES = 1 * 1024 * 1024
SEGMENT_ATTEMPTS = 12
SEGMENT_RETRY_BACKOFF_S = 2
FORCE_IDENTITY_ENCODING = True
DOWNLOAD_ATTEMPTS = 8
RETRY_BACKOFF_S = 10

# Disk-space guard:
# archives + extracted GeoTIFFs + safety/workspace.
REQUIRED_FREE_MULTIPLIER = 2.5
REQUIRED_FREE_EXTRA_GIB = 10.0

# Small distributed spot sample only. Production later performs block-level
# raw-domain checks at every requested location.
SPOT_WINDOWS_PER_RASTER = 16
SPOT_WINDOW_SIZE = 64

EXPECTED_RASTER = {
    "crs_epsg": 5070,
    "pixel_size_m": 30.0,
    "band_count": 1,
    "dtype": "int16",

    # TIFF metadata tag observed consistently on all 10 archive GeoTIFFs.
    "allowed_native_nodata_values": (
        32767.0,
    ),
}

# Pixel-domain sentinel also observed in all 10 archive products.
# It is outside every valid product domain and matches the earlier
# LANDFIRE service-side NoData representation.
ARCHIVE_EMBEDDED_NODATA_SENTINELS = (
    -9999.0,
)

ANALYSIS_NODATA_POLICY = (
    "mask_src_nodata_and_embedded_sentinels_to_nan"
)

print("=" * 100)
print("RUNNING:", NOTEBOOK_BUILD_ID)
print("=" * 100)
print("DOWNLOAD_ARCHIVES:", DOWNLOAD_ARCHIVES)
print("RUN_FULL_ZIP_CRC_TEST:", RUN_FULL_ZIP_CRC_TEST)

## V2 correction — fixed-size segmented Range recovery

The LF2019 Limited Help Desk server repeatedly closed the connection during
the final few MiB of a large resumed HTTP Range response.

V2 preserves the existing `.zip.part` file and changes only the downloader:

- fixed **1 MiB** byte ranges;
- each range is written to a temporary `.segment` file first;
- a segment is appended to `.part` only after:
  - HTTP 206,
  - exact `Content-Range`,
  - exact byte count;
- a failed segment is retried independently;
- already downloaded GiB are never discarded;
- completed archives are still skipped;
- final archive size, SHA-256, ZIP CRC, and GeoTIFF validation remain unchanged.

This is slower in request count but substantially more robust against the
observed USGS server behavior.


## V3 correction — missing `re` import

V2 introduced `CONTENT_RANGE_RE = re.compile(...)` for strict HTTP
`Content-Range` validation but omitted `import re` from the imports cell.

V3 adds only that missing import. Download paths, `.zip.part` reuse,
1 MiB segmented Range recovery, and all downstream validation remain unchanged.


## V4 correction — preserve partial bytes inside each HTTP Range segment

V3 requested an exact 1 MiB range and accepted it only when the whole range
arrived in one connection. If the USGS server delivered 400–700 KiB and then
closed the connection, V3 deleted that useful partial segment and restarted
the same 1 MiB from byte zero.

V4 changes only segment recovery:

1. `.segment` is **preserved** after an interrupted response;
2. on retry, the notebook measures the validated local segment length;
3. the next `Range` begins at `segment_start + bytes_already_received`;
4. `Content-Range` is checked against that exact remaining interval;
5. the completed 1 MiB segment is appended to the archive `.part` only after
   its final byte count is exact;
6. archive-level `.zip.part` reuse remains unchanged.

Thus every successfully received byte now contributes to forward progress,
even when the server repeatedly closes connections mid-response.


## V5 correction — archive-native NoData contract

The downloaded LF2019L (`210`) and LF2020 (`220`) full-extent GeoTIFFs
consistently declare **`32767`** as their embedded NoData value.

Earlier streamed/ImageServer extraction used `-9999`, so V4 incorrectly
hard-coded `-9999` as the only acceptable NoData representation.

V5 separates source-native representation from analysis semantics:

- archived GeoTIFF native NoData: `32767`;
- streamed/ImageServer NoData may differ;
- any source-declared NoData is masked before scientific feature computation;
- the analysis missing representation is `NaN`;
- `32767` must never enter FBFM40/canopy value-domain statistics.

This changes no frozen feature definition; it only corrects source ingestion.


## V6 correction — restore frozen upstream inventory filename

V5 accidentally renamed the upstream LF2020 inventory input from the actual
04D1F V2 artifact:

`LANDFIRE_ARCHIVE_RECOMMENDED_SELECTION_V2.csv`

to a nonexistent V5 filename.

V6 restores the upstream V2 input path. All V5 NoData/source-ingestion fixes
remain unchanged.


## V7 correction — dual missing-value representation in archive GeoTIFFs

The full-extent LF2019L and LF2020 GeoTIFFs use two missing-value
representations simultaneously:

- TIFF metadata (`src.nodata`) declares **32767**;
- sampled pixel arrays also contain **-9999**, which is outside every valid
  thematic domain and is the same sentinel exposed by earlier LANDFIRE
  service extraction.

Therefore the scientifically correct ingestion rule is:

`missing = (value == src.nodata) OR (value == -9999)`

and all such pixels are normalized to `NaN` before feature computation.

V7 keeps these concepts separate:

- **declared native NoData tag:** 32767
- **embedded archive sentinel:** -9999
- **analysis missing representation:** NaN

Neither 32767 nor -9999 is ever allowed to enter FBFM40 or canopy statistics.
The frozen 04D1C V5 feature architecture is unchanged.


## 1. Dependencies and session

In [ ]:
required = {
    "requests": "requests",
    "pandas": "pandas",
    "numpy": "numpy",
    "tqdm": "tqdm",
    "rasterio": "rasterio",
}

for module, package in required.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                package,
            ]
        )

from datetime import datetime, timezone
from urllib.parse import urlparse
import hashlib
import json
import math
import os
import random
import re
import time
import zipfile

import numpy as np
import pandas as pd
import requests
import rasterio
from rasterio.crs import CRS
from rasterio.windows import Window
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 250)
pd.set_option("display.max_rows", 500)
pd.set_option("display.width", 420)

session = requests.Session()

session.headers.update({
    "User-Agent": (
        "UCSD-wildfire-fireline-effectiveness/"
        "04D1G-LANDFIRE-archive-acquisition"
    )
})

## 2. Freeze the 10 reviewed archive selections

In [ ]:
lf2020 = pd.read_csv(
    LF2020_SELECTION_PATH
)

lf2019l = pd.read_csv(
    LF2019L_PLAN_PATH
)

# ------------------------------------------------------------------
# LF2020: V2 unique top candidate.
# ------------------------------------------------------------------
lf2020 = lf2020[
    (
        lf2020["version_key"]
        == "LF2020"
    )
    & (
        lf2020["product"].isin(
            REQUIRED_PRODUCTS
        )
    )
    & (
        lf2020["selection_status"]
        == "UNIQUE_TOP_CANDIDATE"
    )
].copy()

lf2020_plan = pd.DataFrame({
    "version_key": lf2020["version_key"],
    "product": lf2020["product"],
    "archive_name": lf2020["recommended_name"],
    "archive_url": lf2020["recommended_url"],
    "expected_size_bytes": pd.to_numeric(
        lf2020["published_size_bytes"],
        errors="coerce",
    ),
    "source_etag": None,
    "source_kind": "USGS_ScienceBase",
    "source_inventory": str(
        LF2020_SELECTION_PATH
    ),
})

# ------------------------------------------------------------------
# LF2019L: V3 concluded one unique URL/product.
# Deduplicate any repeated candidate rows defensively.
# ------------------------------------------------------------------
lf2019l = lf2019l[
    (
        lf2019l["version_key"]
        == "LF2019L"
    )
    & (
        lf2019l["product"].isin(
            REQUIRED_PRODUCTS
        )
    )
    & (
        lf2019l["download"]
        == True
    )
].copy()

lf2019l = (
    lf2019l.sort_values(
        [
            "product",
            "candidate_url",
        ]
    )
    .drop_duplicates(
        subset=[
            "product",
            "candidate_url",
        ]
    )
)

lf2019l_plan = pd.DataFrame({
    "version_key": lf2019l["version_key"],
    "product": lf2019l["product"],
    "archive_name": lf2019l["candidate_name"],
    "archive_url": lf2019l["candidate_url"],
    "expected_size_bytes": pd.to_numeric(
        lf2019l["content_length"],
        errors="coerce",
    ),
    "source_etag": lf2019l["etag"],
    "source_kind": "USGS_LANDFIRE_HelpDesk",
    "source_inventory": str(
        LF2019L_PLAN_PATH
    ),
})

acquisition_plan = pd.concat(
    [
        lf2019l_plan,
        lf2020_plan,
    ],
    ignore_index=True,
)

acquisition_plan = acquisition_plan.sort_values(
    [
        "version_key",
        "product",
    ]
).reset_index(
    drop=True
)

assert len(
    acquisition_plan
) == 10

assert acquisition_plan[
    [
        "version_key",
        "product",
    ]
].duplicated().sum() == 0

assert set(
    acquisition_plan["product"]
) == set(
    REQUIRED_PRODUCTS
)

for version_key in [
    "LF2019L",
    "LF2020",
]:
    assert set(
        acquisition_plan.loc[
            acquisition_plan[
                "version_key"
            ] == version_key,
            "product",
        ]
    ) == set(
        REQUIRED_PRODUCTS
    )

acquisition_plan[
    "local_archive_path"
] = [
    str(
        ARCHIVE_ROOT /
        version_key /
        archive_name
    )
    for version_key, archive_name in zip(
        acquisition_plan["version_key"],
        acquisition_plan["archive_name"],
    )
]

display(
    acquisition_plan
)

acquisition_plan.to_csv(
    OUTPUT_ROOT /
    "LANDFIRE_ARCHIVE_ACQUISITION_PLAN_V7.csv",
    index=False,
)

print(
    "Frozen archive URLs:",
    acquisition_plan[
        "archive_url"
    ].nunique(),
)

assert acquisition_plan[
    "archive_url"
].nunique() == 10

## 3. Size and disk-space guard

In [ ]:
def human_bytes(n):
    if pd.isna(
        n
    ):
        return None

    value = float(
        n
    )

    for unit in [
        "B",
        "KiB",
        "MiB",
        "GiB",
        "TiB",
    ]:
        if value < 1024:
            return f"{value:.2f} {unit}"

        value /= 1024.0

    return f"{value:.2f} PiB"


expected_archive_bytes = int(
    acquisition_plan[
        "expected_size_bytes"
    ]
    .dropna()
    .sum()
)

usage = shutil.disk_usage(
    PROJECT_ROOT
)

free_bytes = int(
    usage.free
)

required_free_bytes = int(
    expected_archive_bytes
    * REQUIRED_FREE_MULTIPLIER
    + REQUIRED_FREE_EXTRA_GIB
    * 1024**3
)

print(
    "Expected 10-archive download:",
    human_bytes(
        expected_archive_bytes
    ),
)

print(
    "Current free disk:",
    human_bytes(
        free_bytes
    ),
)

print(
    "Required free-space guard:",
    human_bytes(
        required_free_bytes
    ),
)

if free_bytes < required_free_bytes:
    raise RuntimeError(
        "Insufficient free space for archive + extraction safety guard."
    )

print(
    "DISK-SPACE GUARD: PASS"
)

## 4. Source HEAD metadata

In [ ]:
def head_source(
    url,
):
    try:
        r = session.head(
            url,
            allow_redirects=True,
            timeout=REQUEST_TIMEOUT_S,
        )

        return {
            "head_status": r.status_code,
            "head_final_url": r.url,
            "head_content_length": pd.to_numeric(
                r.headers.get(
                    "Content-Length"
                ),
                errors="coerce",
            ),
            "head_etag": r.headers.get(
                "ETag"
            ),
            "head_last_modified": r.headers.get(
                "Last-Modified"
            ),
            "head_accept_ranges": r.headers.get(
                "Accept-Ranges"
            ),
            "head_content_type": r.headers.get(
                "Content-Type"
            ),
            "head_error": None,
        }

    except Exception as exc:
        return {
            "head_status": np.nan,
            "head_final_url": None,
            "head_content_length": np.nan,
            "head_etag": None,
            "head_last_modified": None,
            "head_accept_ranges": None,
            "head_content_type": None,
            "head_error": (
                f"{type(exc).__name__}: "
                f"{str(exc)[:1000]}"
            ),
        }


head_rows = []

for _, r in acquisition_plan.iterrows():
    print(
        "HEAD:",
        r["version_key"],
        r["product"],
        r["archive_name"],
    )

    h = head_source(
        r["archive_url"]
    )

    head_rows.append({
        **r.to_dict(),
        **h,
    })


source_head = pd.DataFrame(
    head_rows
)

source_head[
    "expected_vs_head_size_match"
] = [
    (
        True
        if pd.isna(
            row["expected_size_bytes"]
        )
        or pd.isna(
            row["head_content_length"]
        )
        else int(
            row["expected_size_bytes"]
        )
        == int(
            row["head_content_length"]
        )
    )
    for _, row in source_head.iterrows()
]

display(
    source_head[
        [
            "version_key",
            "product",
            "archive_name",
            "expected_size_bytes",
            "head_content_length",
            "expected_vs_head_size_match",
            "head_status",
            "head_etag",
            "head_accept_ranges",
        ]
    ]
)

source_head.to_csv(
    OUTPUT_ROOT /
    "LANDFIRE_ARCHIVE_SOURCE_HEAD_V7.csv",
    index=False,
)

## 5. Resume-safe archive downloader

Completed archives are skipped when their local byte count matches the
expected/source byte count.

Interrupted downloads remain as `*.part`.

When a server honors `Range`, V1 resumes from the existing `.part` byte offset.
If the server ignores the range and returns `200`, V1 safely restarts that one
archive from byte zero.

In [ ]:

CONTENT_RANGE_RE = re.compile(
    r"^bytes\s+(\d+)-(\d+)/(\d+|\*)$",
    re.I,
)


def expected_remote_size(
    plan_row,
    head_row,
):
    values = [
        plan_row.get(
            "expected_size_bytes"
        ),
        head_row.get(
            "head_content_length"
        ),
    ]

    for value in values:
        if pd.notna(
            value
        ):
            return int(
                value
            )

    return None


def parse_content_range(
    value,
):
    if not value:
        return None

    m = CONTENT_RANGE_RE.match(
        str(
            value
        ).strip()
    )

    if not m:
        return None

    start = int(
        m.group(1)
    )

    end = int(
        m.group(2)
    )

    total = (
        None
        if m.group(3) == "*"
        else int(
            m.group(3)
        )
    )

    return (
        start,
        end,
        total,
    )


def fetch_exact_range_segment(
    url,
    start,
    end,
    segment_path,
    expected_total=None,
):
    """
    Complete one closed archive interval [start, end].

    Unlike V3, an interrupted response does NOT discard bytes already written
    to `.segment`. The next attempt resumes at:

        start + current_segment_size

    The segment is returned only after its exact final size is verified.
    """
    segment_path = Path(
        segment_path
    )

    requested_bytes = (
        end
        - start
        + 1
    )

    # Preserve any partial segment left by the immediately preceding attempt.
    existing_segment_bytes = (
        segment_path.stat().st_size
        if segment_path.exists()
        else 0
    )

    if existing_segment_bytes > requested_bytes:
        segment_path.unlink(
            missing_ok=True
        )
        existing_segment_bytes = 0

    if existing_segment_bytes == requested_bytes:
        return {
            "attempts": 0,
            "bytes": requested_bytes,
            "content_range_total": expected_total,
            "subrequests": 0,
            "recovered_partial_bytes": requested_bytes,
        }

    last_error = None
    total_subrequests = 0
    initial_existing = existing_segment_bytes

    for attempt in range(
        1,
        SEGMENT_ATTEMPTS + 1,
    ):
        current_size = (
            segment_path.stat().st_size
            if segment_path.exists()
            else 0
        )

        if current_size == requested_bytes:
            return {
                "attempts": attempt - 1,
                "bytes": requested_bytes,
                "content_range_total": expected_total,
                "subrequests": total_subrequests,
                "recovered_partial_bytes": (
                    requested_bytes
                    - initial_existing
                ),
            }

        if current_size > requested_bytes:
            segment_path.unlink(
                missing_ok=True
            )
            current_size = 0
            initial_existing = 0

        remaining_start = (
            start
            + current_size
        )

        remaining_end = end

        headers = {
            "Range": (
                f"bytes={remaining_start}-{remaining_end}"
            ),
        }

        if FORCE_IDENTITY_ENCODING:
            headers[
                "Accept-Encoding"
            ] = "identity"

        total_subrequests += 1

        try:
            r = session.get(
                url,
                headers=headers,
                stream=True,
                allow_redirects=True,
                timeout=REQUEST_TIMEOUT_S,
            )

            if r.status_code != 206:
                status = r.status_code
                r.close()

                raise RuntimeError(
                    f"Server did not honor exact range "
                    f"{remaining_start}-{remaining_end}; HTTP {status}"
                )

            parsed = parse_content_range(
                r.headers.get(
                    "Content-Range"
                )
            )

            if parsed is None:
                r.close()

                raise RuntimeError(
                    "Missing/invalid Content-Range on HTTP 206 response."
                )

            got_start, got_end, got_total = parsed

            if (
                got_start != remaining_start
                or got_end != remaining_end
            ):
                r.close()

                raise RuntimeError(
                    f"Content-Range mismatch: requested "
                    f"{remaining_start}-{remaining_end}, "
                    f"got {got_start}-{got_end}"
                )

            if (
                expected_total is not None
                and got_total is not None
                and got_total != expected_total
            ):
                r.close()

                raise RuntimeError(
                    f"Remote total changed: expected "
                    f"{expected_total}, got {got_total}"
                )

            before = current_size

            # Append directly to the temporary segment. If iter_content raises
            # ChunkedEncodingError, bytes yielded before that exception remain
            # on disk and are reused on the next attempt.
            with open(
                segment_path,
                "ab",
            ) as f:
                try:
                    for chunk in r.iter_content(
                        chunk_size=128 * 1024
                    ):
                        if not chunk:
                            continue

                        f.write(
                            chunk
                        )

                        f.flush()

                        now = (
                            segment_path.stat().st_size
                        )

                        if now > requested_bytes:
                            raise RuntimeError(
                                "Range response exceeded target segment size."
                            )

                finally:
                    r.close()

            after = (
                segment_path.stat().st_size
            )

            if after < before:
                raise RuntimeError(
                    "Temporary segment shrank unexpectedly."
                )

            if after == requested_bytes:
                return {
                    "attempts": attempt,
                    "bytes": requested_bytes,
                    "content_range_total": got_total,
                    "subrequests": total_subrequests,
                    "recovered_partial_bytes": (
                        requested_bytes
                        - initial_existing
                    ),
                }

            # A normal clean connection should have delivered the complete
            # requested remainder. If it did not, treat it exactly like an
            # interrupted transfer and continue from `after`.
            last_error = RuntimeError(
                f"Short clean response: segment now "
                f"{after}/{requested_bytes} bytes"
            )

        except Exception as exc:
            last_error = exc

            # IMPORTANT V4 behavior: DO NOT unlink the segment here.
            saved = (
                segment_path.stat().st_size
                if segment_path.exists()
                else 0
            )

            if saved > requested_bytes:
                segment_path.unlink(
                    missing_ok=True
                )
                saved = 0

            print(
                f"    segment {start}-{end} "
                f"attempt {attempt}/{SEGMENT_ATTEMPTS} interrupted:",
                type(
                    exc
                ).__name__,
                str(
                    exc
                )[:350],
                f"| preserved {saved:,}/{requested_bytes:,} bytes",
            )

        if attempt < SEGMENT_ATTEMPTS:
            time.sleep(
                SEGMENT_RETRY_BACKOFF_S
                * min(
                    attempt,
                    5,
                )
            )

    final_saved = (
        segment_path.stat().st_size
        if segment_path.exists()
        else 0
    )

    if final_saved == requested_bytes:
        return {
            "attempts": SEGMENT_ATTEMPTS,
            "bytes": requested_bytes,
            "content_range_total": expected_total,
            "subrequests": total_subrequests,
            "recovered_partial_bytes": (
                requested_bytes
                - initial_existing
            ),
        }

    raise RuntimeError(
        f"Exact range segment {start}-{end} incomplete after "
        f"{SEGMENT_ATTEMPTS} attempts; preserved "
        f"{final_saved}/{requested_bytes} bytes. "
        f"Last error: {last_error}"
    )


def download_with_resume(
    url,
    final_path,
    expected_size=None,
):
    """
    Resume-safe segmented archive acquisition.

    V2 deliberately fetches fixed-size byte intervals instead of requesting
    the entire remaining tail in one connection.
    """
    final_path = Path(
        final_path
    )

    final_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    part_path = final_path.with_suffix(
        final_path.suffix + ".part"
    )

    segment_path = final_path.with_suffix(
        final_path.suffix + ".segment"
    )

    # Remove only an orphan segment from a PRIOR kernel/run. Within the
    # current fetch call, V4 preserves partial segment bytes across retries.
    segment_path.unlink(
        missing_ok=True
    )

    # --------------------------------------------------------------
    # Completed final file.
    # --------------------------------------------------------------
    if final_path.exists():
        local_size = (
            final_path.stat().st_size
        )

        if (
            expected_size is None
            or local_size == expected_size
        ):
            return {
                "status": "existing_complete",
                "bytes": local_size,
                "resumed_from": 0,
                "attempts": 0,
                "segments_downloaded": 0,
                "segment_retry_count": 0,
                "path": str(
                    final_path
                ),
            }

        bad = final_path.with_suffix(
            final_path.suffix
            + ".size_mismatch"
        )

        bad.unlink(
            missing_ok=True
        )

        final_path.replace(
            bad
        )

    # V2 segmented mode requires known total size.
    if expected_size is None:
        raise RuntimeError(
            "V2 segmented downloader requires a known remote archive size."
        )

    expected_size = int(
        expected_size
    )

    # --------------------------------------------------------------
    # Reuse current V1/V2 .part file exactly as-is.
    # --------------------------------------------------------------
    existing = (
        part_path.stat().st_size
        if (
            RESUME_PARTIAL_DOWNLOADS
            and part_path.exists()
        )
        else 0
    )

    if existing > expected_size:
        bad = part_path.with_suffix(
            part_path.suffix
            + ".oversize"
        )

        bad.unlink(
            missing_ok=True
        )

        part_path.replace(
            bad
        )

        existing = 0

    resumed_from = existing

    # A V1 partial file may already be exactly complete if the last exception
    # happened after the final write but before rename.
    if existing == expected_size:
        os.replace(
            str(
                part_path
            ),
            str(
                final_path
            ),
        )

        return {
            "status": "completed_existing_part",
            "bytes": expected_size,
            "resumed_from": resumed_from,
            "attempts": 0,
            "segments_downloaded": 0,
            "segment_retry_count": 0,
            "path": str(
                final_path
            ),
        }

    progress = tqdm(
        total=expected_size,
        initial=existing,
        unit="B",
        unit_scale=True,
        unit_divisor=1024,
        desc=final_path.name,
        leave=True,
    )

    current = existing
    segments_downloaded = 0
    total_segment_attempts = 0

    try:
        while current < expected_size:

            end = min(
                current
                + RANGE_SEGMENT_BYTES
                - 1,
                expected_size
                - 1,
            )

            seg = fetch_exact_range_segment(
                url,
                current,
                end,
                segment_path,
                expected_total=expected_size,
            )

            segment_bytes = (
                end
                - current
                + 1
            )

            # Append only a completely validated segment.
            with open(
                part_path,
                "ab",
            ) as dst, open(
                segment_path,
                "rb",
            ) as src:
                shutil.copyfileobj(
                    src,
                    dst,
                    length=256 * 1024,
                )

                dst.flush()

            segment_path.unlink(
                missing_ok=True
            )

            new_size = (
                part_path.stat().st_size
            )

            expected_new_size = (
                current
                + segment_bytes
            )

            if new_size != expected_new_size:
                raise RuntimeError(
                    f"Local .part size after segment append "
                    f"{new_size} != expected {expected_new_size}"
                )

            current = new_size
            segments_downloaded += 1
            total_segment_attempts += int(
                seg[
                    "attempts"
                ]
            )

            progress.update(
                segment_bytes
            )

    finally:
        progress.close()

        segment_path.unlink(
            missing_ok=True
        )

    final_part_size = (
        part_path.stat().st_size
    )

    if final_part_size != expected_size:
        raise RuntimeError(
            f"Final .part size {final_part_size} "
            f"!= expected {expected_size}"
        )

    os.replace(
        str(
            part_path
        ),
        str(
            final_path
        ),
    )

    segment_retry_count = max(
        0,
        total_segment_attempts
        - segments_downloaded,
    )

    return {
        "status": "downloaded_segmented",
        "bytes": final_path.stat().st_size,
        "resumed_from": resumed_from,
        "attempts": total_segment_attempts,
        "segments_downloaded": segments_downloaded,
        "segment_retry_count": segment_retry_count,
        "path": str(
            final_path
        ),
    }


## 6. Download / resume the 10 archives

In [ ]:
download_rows = []

head_lookup = {
    (
        r[
            "version_key"
        ],
        r[
            "product"
        ],
    ): r
    for _, r in source_head.iterrows()
}

for i, (_, r) in enumerate(
    acquisition_plan.iterrows(),
    start=1,
):
    version_key = r[
        "version_key"
    ]

    product = r[
        "product"
    ]

    h = head_lookup[
        (
            version_key,
            product,
        )
    ]

    expected_size = expected_remote_size(
        r,
        h,
    )

    local_path = Path(
        r[
            "local_archive_path"
        ]
    )

    print()
    print("=" * 100)
    print(
        f"[{i}/10]",
        version_key,
        product,
    )
    print(
        r[
            "archive_name"
        ]
    )
    print("=" * 100)

    if not DOWNLOAD_ARCHIVES:
        result = {
            "status": "download_disabled",
            "bytes": (
                local_path.stat().st_size
                if local_path.exists()
                else 0
            ),
            "resumed_from": 0,
            "attempts": 0,
            "segments_downloaded": 0,
            "segment_retry_count": 0,
            "path": str(
                local_path
            ),
        }

    else:
        result = download_with_resume(
            r[
                "archive_url"
            ],
            local_path,
            expected_size=expected_size,
        )

    download_rows.append({
        "version_key": version_key,
        "product": product,
        "archive_name": r[
            "archive_name"
        ],
        "archive_url": r[
            "archive_url"
        ],
        "expected_size_bytes": expected_size,
        "download_status": result[
            "status"
        ],
        "local_size_bytes": result[
            "bytes"
        ],
        "resumed_from_bytes": result[
            "resumed_from"
        ],
        "download_attempts": result[
            "attempts"
        ],
        "segments_downloaded": result.get(
            "segments_downloaded",
            0,
        ),
        "segment_retry_count": result.get(
            "segment_retry_count",
            0,
        ),
        "local_archive_path": result[
            "path"
        ],
    })

    pd.DataFrame(
        download_rows
    ).to_csv(
        OUTPUT_ROOT /
        "LANDFIRE_ARCHIVE_DOWNLOAD_STATUS_INCREMENTAL_V7.csv",
        index=False,
    )


download_status = pd.DataFrame(
    download_rows
)

download_status[
    "size_match"
] = [
    (
        True
        if pd.isna(
            row[
                "expected_size_bytes"
            ]
        )
        else int(
            row[
                "local_size_bytes"
            ]
        )
        == int(
            row[
                "expected_size_bytes"
            ]
        )
    )
    for _, row in download_status.iterrows()
]

display(
    download_status
)

assert download_status[
    "size_match"
].all()

download_status.to_csv(
    OUTPUT_ROOT /
    "LANDFIRE_ARCHIVE_DOWNLOAD_STATUS_V7.csv",
    index=False,
)

print(
    "ALL ARCHIVE BYTE COUNTS: PASS"
)

## 7. SHA-256 and ZIP integrity

In [ ]:
def sha256_file(
    path,
    chunk_bytes=8 * 1024 * 1024,
):
    h = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as f:
        while True:
            chunk = f.read(
                chunk_bytes
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


integrity_rows = []

for i, (_, r) in enumerate(
    download_status.iterrows(),
    start=1,
):
    path = Path(
        r[
            "local_archive_path"
        ]
    )

    print()
    print(
        f"INTEGRITY {i}/10:",
        r[
            "version_key"
        ],
        r[
            "product"
        ],
        path.name,
    )

    sha256 = sha256_file(
        path
    )

    with zipfile.ZipFile(
        path,
        "r",
        allowZip64=True,
    ) as zf:
        bad_member = (
            zf.testzip()
            if RUN_FULL_ZIP_CRC_TEST
            else None
        )

        member_count = len(
            zf.infolist()
        )

    integrity_rows.append({
        "version_key": r[
            "version_key"
        ],
        "product": r[
            "product"
        ],
        "archive_name": path.name,
        "local_archive_path": str(
            path
        ),
        "local_size_bytes": path.stat().st_size,
        "sha256": sha256,
        "zip_open_pass": True,
        "zip_crc_test_run": (
            RUN_FULL_ZIP_CRC_TEST
        ),
        "zip_crc_pass": (
            bad_member is None
        ),
        "bad_zip_member": bad_member,
        "zip_member_count": member_count,
    })

    pd.DataFrame(
        integrity_rows
    ).to_csv(
        OUTPUT_ROOT /
        "LANDFIRE_ARCHIVE_INTEGRITY_INCREMENTAL_V7.csv",
        index=False,
    )


integrity = pd.DataFrame(
    integrity_rows
)

display(
    integrity
)

assert integrity[
    "zip_open_pass"
].all()

if RUN_FULL_ZIP_CRC_TEST:
    assert integrity[
        "zip_crc_pass"
    ].all()

integrity.to_csv(
    OUTPUT_ROOT /
    "LANDFIRE_ARCHIVE_INTEGRITY_V7.csv",
    index=False,
)

print(
    "ZIP INTEGRITY: PASS"
)

## 8. ZIP member inventory

In [ ]:
member_rows = []

for _, r in integrity.iterrows():

    path = Path(
        r[
            "local_archive_path"
        ]
    )

    with zipfile.ZipFile(
        path,
        "r",
        allowZip64=True,
    ) as zf:

        for info in zf.infolist():

            suffix = Path(
                info.filename
            ).suffix.lower()

            member_rows.append({
                "version_key": r[
                    "version_key"
                ],
                "product": r[
                    "product"
                ],
                "archive_name": path.name,
                "member_name": info.filename,
                "member_suffix": suffix,
                "compressed_size": info.compress_size,
                "uncompressed_size": info.file_size,
                "crc32": f"{info.CRC:08x}",
                "is_directory": info.is_dir(),
                "is_geotiff": (
                    suffix
                    in {
                        ".tif",
                        ".tiff",
                    }
                ),
                "is_xml": (
                    suffix == ".xml"
                ),
                "is_csv": (
                    suffix == ".csv"
                ),
            })


members = pd.DataFrame(
    member_rows
)

member_summary = (
    members.groupby(
        [
            "version_key",
            "product",
        ],
        as_index=False,
    )
    .agg(
        members=(
            "member_name",
            "size",
        ),
        geotiffs=(
            "is_geotiff",
            "sum",
        ),
        xml_files=(
            "is_xml",
            "sum",
        ),
        csv_files=(
            "is_csv",
            "sum",
        ),
        total_uncompressed_bytes=(
            "uncompressed_size",
            "sum",
        ),
    )
)

display(
    member_summary
)

members.to_csv(
    OUTPUT_ROOT /
    "LANDFIRE_ARCHIVE_MEMBER_INVENTORY_V7.csv",
    index=False,
)

member_summary.to_csv(
    OUTPUT_ROOT /
    "LANDFIRE_ARCHIVE_MEMBER_SUMMARY_V7.csv",
    index=False,
)

# One raw product GeoTIFF per archive is required.
assert (
    member_summary[
        "geotiffs"
    ] == 1
).all()

print(
    "ONE GEOTIFF PER ARCHIVE: PASS"
)

## 9. Safe selective extraction

Only these members are extracted:

- `.tif` / `.tiff`
- `.xml`
- `.csv`

No arbitrary archive path is trusted. Each destination is resolved and checked
to remain under its designated `version/product` extraction directory.

In [ ]:
def safe_extract_member(
    zf,
    info,
    dest_root,
):
    dest_root = Path(
        dest_root
    ).resolve()

    # Preserve only the member basename. Each archive has its own
    # version/product destination, avoiding path traversal and unnecessary
    # nested release directories.
    dest = (
        dest_root /
        Path(
            info.filename
        ).name
    ).resolve()

    if dest.parent != dest_root:
        raise RuntimeError(
            f"Unsafe ZIP member path: {info.filename}"
        )

    if dest.exists():
        # Existing extracted file is accepted only when its byte count matches.
        if dest.stat().st_size == info.file_size:
            return dest, "existing_complete"

        bad = dest.with_suffix(
            dest.suffix
            + ".size_mismatch"
        )

        if bad.exists():
            bad.unlink()

        dest.replace(
            bad
        )

    tmp = dest.with_suffix(
        dest.suffix + ".part"
    )

    tmp.unlink(
        missing_ok=True
    )

    with zf.open(
        info,
        "r",
    ) as src, open(
        tmp,
        "wb",
    ) as dst:
        shutil.copyfileobj(
            src,
            dst,
            length=8 * 1024 * 1024,
        )

    if tmp.stat().st_size != info.file_size:
        raise RuntimeError(
            f"Extracted size mismatch: {info.filename}"
        )

    os.replace(
        str(
            tmp
        ),
        str(
            dest
        ),
    )

    return dest, "extracted"


extract_rows = []

for _, r in integrity.iterrows():

    archive = Path(
        r[
            "local_archive_path"
        ]
    )

    dest_root = (
        EXTRACT_ROOT /
        r[
            "version_key"
        ] /
        r[
            "product"
        ]
    )

    dest_root.mkdir(
        parents=True,
        exist_ok=True,
    )

    print(
        "EXTRACT:",
        r[
            "version_key"
        ],
        r[
            "product"
        ],
    )

    with zipfile.ZipFile(
        archive,
        "r",
        allowZip64=True,
    ) as zf:

        selected = [
            info
            for info in zf.infolist()
            if (
                not info.is_dir()
                and Path(
                    info.filename
                ).suffix.lower()
                in {
                    ".tif",
                    ".tiff",
                    ".xml",
                    ".csv",
                }
            )
        ]

        for info in selected:

            path, status = (
                safe_extract_member(
                    zf,
                    info,
                    dest_root,
                )
            )

            extract_rows.append({
                "version_key": r[
                    "version_key"
                ],
                "product": r[
                    "product"
                ],
                "archive_name": archive.name,
                "member_name": info.filename,
                "member_size": info.file_size,
                "local_extracted_path": str(
                    path
                ),
                "extract_status": status,
            })


extracted = pd.DataFrame(
    extract_rows
)

display(
    extracted
)

extracted.to_csv(
    OUTPUT_ROOT /
    "LANDFIRE_ARCHIVE_EXTRACTED_FILES_V7.csv",
    index=False,
)

## 10. GeoTIFF metadata validation

In [ ]:
tif_rows = []

for (
    version_key,
    product
), g in extracted.groupby(
    [
        "version_key",
        "product",
    ],
):

    tif_files = g[
        g[
            "local_extracted_path"
        ].str.lower().str.endswith(
            (
                ".tif",
                ".tiff",
            )
        )
    ]

    if len(
        tif_files
    ) != 1:
        raise RuntimeError(
            f"{version_key}/{product}: expected exactly 1 GeoTIFF, "
            f"found {len(tif_files)}"
        )

    path = Path(
        tif_files.iloc[
            0
        ][
            "local_extracted_path"
        ]
    )

    with rasterio.open(
        path
    ) as src:

        crs_epsg = (
            src.crs.to_epsg()
            if src.crs is not None
            else None
        )

        row = {
            "version_key": version_key,
            "product": product,
            "tif_path": str(
                path
            ),
            "width": src.width,
            "height": src.height,
            "band_count": src.count,
            "dtype": src.dtypes[
                0
            ],
            "crs": str(
                src.crs
            ),
            "crs_epsg": crs_epsg,
            "nodata": src.nodata,
            "transform_a": src.transform.a,
            "transform_b": src.transform.b,
            "transform_c": src.transform.c,
            "transform_d": src.transform.d,
            "transform_e": src.transform.e,
            "transform_f": src.transform.f,
            "pixel_x_m": abs(
                src.transform.a
            ),
            "pixel_y_m": abs(
                src.transform.e
            ),
            "left": src.bounds.left,
            "bottom": src.bounds.bottom,
            "right": src.bounds.right,
            "top": src.bounds.top,
            "compression": (
                src.compression.value
                if src.compression
                is not None
                else None
            ),
            "tiled": src.is_tiled,
        }

    row[
        "crs_pass"
    ] = (
        row[
            "crs_epsg"
        ]
        == EXPECTED_RASTER[
            "crs_epsg"
        ]
    )

    row[
        "pixel_size_pass"
    ] = (
        abs(
            row[
                "pixel_x_m"
            ]
            - EXPECTED_RASTER[
                "pixel_size_m"
            ]
        ) <= 1e-9
        and abs(
            row[
                "pixel_y_m"
            ]
            - EXPECTED_RASTER[
                "pixel_size_m"
            ]
        ) <= 1e-9
    )

    row[
        "band_count_pass"
    ] = (
        row[
            "band_count"
        ]
        == EXPECTED_RASTER[
            "band_count"
        ]
    )

    row[
        "dtype_pass"
    ] = (
        str(
            row[
                "dtype"
            ]
        ).lower()
        == EXPECTED_RASTER[
            "dtype"
        ]
    )

    row[
        "nodata_pass"
    ] = (
        row[
            "nodata"
        ] is not None
        and any(
            float(
                row[
                    "nodata"
                ]
            )
            == float(
                allowed
            )
            for allowed in EXPECTED_RASTER[
                "allowed_native_nodata_values"
            ]
        )
    )

    row[
        "nodata_semantic_policy"
    ] = ANALYSIS_NODATA_POLICY

    tif_rows.append(
        row
    )


raster_metadata = pd.DataFrame(
    tif_rows
)

display(
    raster_metadata[
        [
            "version_key",
            "product",
            "width",
            "height",
            "crs_epsg",
            "pixel_x_m",
            "pixel_y_m",
            "dtype",
            "nodata",
            "crs_pass",
            "pixel_size_pass",
            "band_count_pass",
            "dtype_pass",
            "nodata_pass",
        ]
    ]
)

metadata_pass_cols = [
    "crs_pass",
    "pixel_size_pass",
    "band_count_pass",
    "dtype_pass",
    "nodata_pass",
]

assert raster_metadata[
    metadata_pass_cols
].all().all()

raster_metadata.to_csv(
    OUTPUT_ROOT /
    "LANDFIRE_ARCHIVE_RASTER_METADATA_V7.csv",
    index=False,
)

print(
    "RASTER METADATA CONTRACT: PASS"
)

In [ ]:
nodata_audit = (
    raster_metadata.groupby(
        "version_key",
        as_index=False,
    )
    .agg(
        products=(
            "product",
            "size",
        ),
        unique_native_nodata_values=(
            "nodata",
            lambda s: sorted(
                {
                    float(x)
                    for x in s.dropna()
                }
            ),
        ),
        all_nodata_contract_pass=(
            "nodata_pass",
            "all",
        ),
    )
)

display(nodata_audit)

assert nodata_audit[
    "all_nodata_contract_pass"
].all()

assert all(
    values == [32767.0]
    for values in nodata_audit[
        "unique_native_nodata_values"
    ]
)

nodata_audit.to_csv(
    OUTPUT_ROOT /
    "LANDFIRE_ARCHIVE_NATIVE_NODATA_AUDIT_V7.csv",
    index=False,
)

print("ARCHIVE-NATIVE NODATA CONTRACT: PASS")
print(
    "Declared GeoTIFF NoData tag contract: PASS"
)
print(
    "Embedded pixel sentinel(s):",
    ARCHIVE_EMBEDDED_NODATA_SENTINELS,
)
print(
    "Analysis policy:",
    ANALYSIS_NODATA_POLICY,
)


## 11. Within-vintage raster geometry alignment

In [ ]:
geometry_rows = []

for version_key, g in raster_metadata.groupby(
    "version_key"
):

    ref = g.sort_values(
        "product"
    ).iloc[
        0
    ]

    geometry_columns = [
        "width",
        "height",
        "crs_epsg",
        "transform_a",
        "transform_b",
        "transform_c",
        "transform_d",
        "transform_e",
        "transform_f",
        "left",
        "bottom",
        "right",
        "top",
    ]

    for _, r in g.iterrows():

        exact = True
        differences = []

        for col in geometry_columns:

            a = ref[
                col
            ]

            b = r[
                col
            ]

            if isinstance(
                a,
                (float, np.floating),
            ) or isinstance(
                b,
                (float, np.floating),
            ):
                same = bool(
                    np.isclose(
                        float(
                            a
                        ),
                        float(
                            b
                        ),
                        atol=1e-9,
                        rtol=0,
                    )
                )
            else:
                same = (
                    a == b
                )

            if not same:
                exact = False
                differences.append(
                    col
                )

        geometry_rows.append({
            "version_key": version_key,
            "reference_product": ref[
                "product"
            ],
            "comparison_product": r[
                "product"
            ],
            "geometry_identical": exact,
            "differences": ";".join(
                differences
            ),
        })


geometry_alignment = pd.DataFrame(
    geometry_rows
)

display(
    geometry_alignment
)

geometry_alignment.to_csv(
    OUTPUT_ROOT /
    "LANDFIRE_ARCHIVE_RASTER_GEOMETRY_ALIGNMENT_V7.csv",
    index=False,
)

# Do not force identity if LANDFIRE full-extent products use different outer
# padding. Joint canopy production later requires physical sampled-cell
# alignment, just as in 04D1D/04D1E.
print(
    "Geometry-identical rows:",
    int(
        geometry_alignment[
            "geometry_identical"
        ].sum()
    ),
    "/",
    len(
        geometry_alignment
    ),
)

## 12. Distributed raw-value spot checks

In [ ]:

FBFM40_VALID_CODES = (
    {91, 92, 93, 98, 99}
    | set(
        range(
            101,
            110,
        )
    )
    | set(
        range(
            121,
            125,
        )
    )
    | set(
        range(
            141,
            150,
        )
    )
    | set(
        range(
            161,
            166,
        )
    )
    | set(
        range(
            181,
            190,
        )
    )
    | set(
        range(
            201,
            205,
        )
    )
)

assert len(
    FBFM40_VALID_CODES
) == 45


def missing_mask(
    values,
    declared_nodata,
):
    """
    Archive missing-value contract.

    Missing if:
      1. non-finite, OR
      2. equals raster-declared src.nodata, OR
      3. equals a documented embedded archive sentinel (-9999).
    """
    values = np.asarray(
        values
    )

    mask = ~np.isfinite(
        values
    )

    if (
        declared_nodata is not None
        and np.isfinite(
            declared_nodata
        )
    ):
        mask |= (
            values
            == declared_nodata
        )

    for sentinel in (
        ARCHIVE_EMBEDDED_NODATA_SENTINELS
    ):
        mask |= (
            values
            == sentinel
        )

    return mask


def raw_domain_ok(
    product,
    values,
    declared_nodata,
):
    values = np.asarray(
        values
    )

    mask = missing_mask(
        values,
        declared_nodata,
    )

    values = values[
        ~mask
    ]

    if not len(
        values
    ):
        return True

    if product == "FBFM40":
        # FBFM40 is categorical: do not average or scale raw codes.
        return bool(
            np.isin(
                values.astype(
                    np.int64
                ),
                list(
                    FBFM40_VALID_CODES
                ),
            ).all()
        )

    if product == "CC":
        return bool(
            (
                (
                    values >= 0
                )
                & (
                    values <= 100
                )
            ).all()
        )

    if product == "CH":
        return bool(
            (
                (
                    values >= 0
                )
                & (
                    values <= 510
                )
            ).all()
        )

    if product == "CBH":
        return bool(
            (
                (
                    values >= 0
                )
                & (
                    values <= 100
                )
            ).all()
        )

    if product == "CBD":
        return bool(
            (
                (
                    values >= 0
                )
                & (
                    values <= 45
                )
            ).all()
        )

    raise KeyError(
        product
    )


spot_rows = []

rng = np.random.default_rng(
    20260825
)

for _, r in raster_metadata.iterrows():

    path = Path(
        r[
            "tif_path"
        ]
    )

    product = r[
        "product"
    ]

    collected = []

    with rasterio.open(
        path
    ) as src:

        # Re-read directly from the source metadata for provenance.
        declared_nodata = (
            float(
                src.nodata
            )
            if src.nodata is not None
            else None
        )

        max_row = max(
            0,
            src.height
            - SPOT_WINDOW_SIZE
        )

        max_col = max(
            0,
            src.width
            - SPOT_WINDOW_SIZE
        )

        for _ in range(
            SPOT_WINDOWS_PER_RASTER
        ):

            row_off = (
                int(
                    rng.integers(
                        0,
                        max_row + 1,
                    )
                )
                if max_row > 0
                else 0
            )

            col_off = (
                int(
                    rng.integers(
                        0,
                        max_col + 1,
                    )
                )
                if max_col > 0
                else 0
            )

            arr = src.read(
                1,
                window=Window(
                    col_off,
                    row_off,
                    min(
                        SPOT_WINDOW_SIZE,
                        src.width
                        - col_off,
                    ),
                    min(
                        SPOT_WINDOW_SIZE,
                        src.height
                        - row_off,
                    ),
                ),
            )

            collected.append(
                arr.reshape(
                    -1
                )
            )

    sampled = np.concatenate(
        collected
    )

    miss = missing_mask(
        sampled,
        declared_nodata,
    )

    valid = sampled[
        ~miss
    ]

    count_declared_nodata = (
        int(
            np.sum(
                sampled
                == declared_nodata
            )
        )
        if declared_nodata
        is not None
        else 0
    )

    sentinel_counts = {
        str(
            int(
                sentinel
            )
        ): int(
            np.sum(
                sampled
                == sentinel
            )
        )
        for sentinel in (
            ARCHIVE_EMBEDDED_NODATA_SENTINELS
        )
    }

    spot_rows.append({
        "version_key": r[
            "version_key"
        ],
        "product": product,
        "sampled_values": int(
            len(
                sampled
            )
        ),
        "missing_values_total": int(
            miss.sum()
        ),
        "valid_nonmissing_values": int(
            len(
                valid
            )
        ),
        "declared_native_nodata_value": (
            declared_nodata
        ),
        "declared_nodata_pixel_count": (
            count_declared_nodata
        ),
        "embedded_sentinel_counts": (
            json.dumps(
                sentinel_counts,
                sort_keys=True,
            )
        ),
        "sample_min_after_missing_mask": (
            float(
                np.min(
                    valid
                )
            )
            if len(
                valid
            )
            else np.nan
        ),
        "sample_max_after_missing_mask": (
            float(
                np.max(
                    valid
                )
            )
            if len(
                valid
            )
            else np.nan
        ),
        "spot_domain_pass": raw_domain_ok(
            product,
            sampled,
            declared_nodata,
        ),
    })


spot_checks = pd.DataFrame(
    spot_rows
)

display(
    spot_checks
)

assert spot_checks[
    "spot_domain_pass"
].all()

# Explicitly verify that -9999 is being treated as missing rather than
# surviving into the scientific value domain whenever encountered.
for _, row in spot_checks.iterrows():

    counts = json.loads(
        row[
            "embedded_sentinel_counts"
        ]
    )

    if counts.get(
        "-9999",
        0,
    ) > 0:
        assert (
            row[
                "sample_min_after_missing_mask"
            ]
            > -9999
            or pd.isna(
                row[
                    "sample_min_after_missing_mask"
                ]
            )
        )


spot_checks.to_csv(
    OUTPUT_ROOT /
    "LANDFIRE_ARCHIVE_RASTER_SPOT_DOMAIN_QA_V7.csv",
    index=False,
)

print(
    "DISTRIBUTED RAW-VALUE SPOT QA: PASS"
)

print(
    "Missing-value policy:",
    ANALYSIS_NODATA_POLICY,
)

print(
    "Declared TIFF NoData:",
    EXPECTED_RASTER[
        "allowed_native_nodata_values"
    ],
)

print(
    "Embedded archive sentinels:",
    ARCHIVE_EMBEDDED_NODATA_SENTINELS,
)


## 13. Final provenance manifest

In [ ]:
manifest = (
    acquisition_plan.merge(
        source_head[
            [
                "version_key",
                "product",
                "head_status",
                "head_final_url",
                "head_content_length",
                "head_etag",
                "head_last_modified",
                "head_accept_ranges",
            ]
        ],
        on=[
            "version_key",
            "product",
        ],
        how="left",
        validate="one_to_one",
    )
    .merge(
        download_status[
            [
                "version_key",
                "product",
                "download_status",
                "local_size_bytes",
                "local_archive_path",
                "size_match",
            ]
        ],
        on=[
            "version_key",
            "product",
        ],
        how="left",
        validate="one_to_one",
        suffixes=(
            "",
            "_download",
        ),
    )
    .merge(
        integrity[
            [
                "version_key",
                "product",
                "sha256",
                "zip_open_pass",
                "zip_crc_test_run",
                "zip_crc_pass",
                "zip_member_count",
            ]
        ],
        on=[
            "version_key",
            "product",
        ],
        how="left",
        validate="one_to_one",
    )
    .merge(
        member_summary[
            [
                "version_key",
                "product",
                "geotiffs",
                "xml_files",
                "csv_files",
                "total_uncompressed_bytes",
            ]
        ],
        on=[
            "version_key",
            "product",
        ],
        how="left",
        validate="one_to_one",
    )
    .merge(
        raster_metadata[
            [
                "version_key",
                "product",
                "tif_path",
                "width",
                "height",
                "crs_epsg",
                "pixel_x_m",
                "pixel_y_m",
                "dtype",
                "nodata",
                "crs_pass",
                "pixel_size_pass",
                "band_count_pass",
                "dtype_pass",
                "nodata_pass",
            ]
        ],
        on=[
            "version_key",
            "product",
        ],
        how="left",
        validate="one_to_one",
    )
    .merge(
        spot_checks[
            [
                "version_key",
                "product",
                "missing_values_total",
                "valid_nonmissing_values",
                "declared_native_nodata_value",
                "declared_nodata_pixel_count",
                "embedded_sentinel_counts",
                "sample_min_after_missing_mask",
                "sample_max_after_missing_mask",
                "spot_domain_pass",
            ]
        ],
        on=[
            "version_key",
            "product",
        ],
        how="left",
        validate="one_to_one",
    )
)

manifest[
    "acquisition_validated"
] = (
    manifest[
        "size_match"
    ].fillna(
        False
    )
    & manifest[
        "zip_open_pass"
    ].fillna(
        False
    )
    & (
        manifest[
            "zip_crc_pass"
        ].fillna(
            False
        )
        if RUN_FULL_ZIP_CRC_TEST
        else True
    )
    & (
        manifest[
            "geotiffs"
        ] == 1
    )
    & manifest[
        [
            "crs_pass",
            "pixel_size_pass",
            "band_count_pass",
            "dtype_pass",
            "nodata_pass",
            "spot_domain_pass",
        ]
    ].fillna(
        False
    ).all(
        axis=1
    )
)

display(
    manifest
)

manifest.to_csv(
    OUTPUT_ROOT /
    "LANDFIRE_ARCHIVE_PROVENANCE_MANIFEST_VALIDATED_V7.csv",
    index=False,
)

assert len(
    manifest
) == 10

assert manifest[
    "acquisition_validated"
].all()

print(
    "ALL 10 ARCHIVES ACQUISITION-VALIDATED: PASS"
)

## 14. Final checks

In [ ]:
checks = pd.DataFrame([
    {
        "check": "exactly_10_reviewed_archive_sources",
        "pass": (
            len(
                acquisition_plan
            ) == 10
            and acquisition_plan[
                "archive_url"
            ].nunique()
            == 10
        ),
    },
    {
        "check": "all_local_archive_sizes_match",
        "pass": bool(
            download_status[
                "size_match"
            ].all()
        ),
    },
    {
        "check": "all_zip_archives_open",
        "pass": bool(
            integrity[
                "zip_open_pass"
            ].all()
        ),
    },
    {
        "check": "all_zip_crc_pass",
        "pass": bool(
            integrity[
                "zip_crc_pass"
            ].all()
        )
        if RUN_FULL_ZIP_CRC_TEST
        else True,
    },
    {
        "check": "one_geotiff_per_archive",
        "pass": bool(
            (
                member_summary[
                    "geotiffs"
                ] == 1
            ).all()
        ),
    },
    {
        "check": "all_raster_metadata_contracts_pass",
        "pass": bool(
            raster_metadata[
                [
                    "crs_pass",
                    "pixel_size_pass",
                    "band_count_pass",
                    "dtype_pass",
                    "nodata_pass",
                ]
            ].all().all()
        ),
    },
    {
        "check": "archive_missing_representation_contract_defined",
        "pass": bool(
            EXPECTED_RASTER[
                "allowed_native_nodata_values"
            ] == (
                32767.0,
            )
            and ARCHIVE_EMBEDDED_NODATA_SENTINELS
            == (
                -9999.0,
            )
            and ANALYSIS_NODATA_POLICY
            == (
                "mask_src_nodata_and_embedded_sentinels_to_nan"
            )
        ),
    },
    {
        "check": "all_distributed_raw_domain_spot_checks_pass",
        "pass": bool(
            spot_checks[
                "spot_domain_pass"
            ].all()
        ),
    },
    {
        "check": "all_10_acquisitions_validated",
        "pass": bool(
            manifest[
                "acquisition_validated"
            ].all()
        ),
    },
])

display(
    checks
)

ACQUISITION_PASS = bool(
    checks[
        "pass"
    ].all()
)

print()
print(
    "04D1G ARCHIVED LANDFIRE ACQUISITION:",
    (
        "PASS / READY FOR ARCHIVED-VINTAGE PRODUCTION"
        if ACQUISITION_PASS
        else "REVIEW REQUIRED"
    ),
)

checks.to_csv(
    OUTPUT_ROOT /
    "LANDFIRE_ARCHIVE_ACQUISITION_CHECKS_V7.csv",
    index=False,
)

## 15. Report

In [ ]:
def tomd(df):
    try:
        return df.to_markdown(
            index=False
        )
    except Exception:
        return df.to_string(
            index=False
        )


size_summary = (
    manifest.groupby(
        "version_key",
        as_index=False,
    )
    .agg(
        products=(
            "product",
            "size",
        ),
        archive_bytes=(
            "local_size_bytes",
            "sum",
        ),
        extracted_uncompressed_bytes=(
            "total_uncompressed_bytes",
            "sum",
        ),
        validated=(
            "acquisition_validated",
            "all",
        ),
    )
)

size_summary[
    "archive_gib"
] = (
    size_summary[
        "archive_bytes"
    ]
    / 1024**3
)

size_summary[
    "extracted_uncompressed_gib"
] = (
    size_summary[
        "extracted_uncompressed_bytes"
    ]
    / 1024**3
)


report = f"""# 04D1G Archived LANDFIRE Acquisition & Validation

Build: `{NOTEBOOK_BUILD_ID}`

Generated UTC: {datetime.now(timezone.utc).isoformat()}

## Acquired source set

{tomd(acquisition_plan)}

## Size summary

{tomd(size_summary)}

## ZIP member summary

{tomd(member_summary)}

## Raster metadata

{tomd(raster_metadata[['version_key','product','width','height','crs_epsg','pixel_x_m','pixel_y_m','dtype','nodata','crs_pass','pixel_size_pass','dtype_pass','nodata_pass']])}

## Missing-value ingestion contract

Declared TIFF NoData: `{EXPECTED_RASTER["allowed_native_nodata_values"]}`

Embedded archive sentinel(s): `{ARCHIVE_EMBEDDED_NODATA_SENTINELS}`

Analysis policy: `{ANALYSIS_NODATA_POLICY}`

## Distributed raw-domain QA

{tomd(spot_checks)}

## Final checks

{tomd(checks)}

## Decision

`{"PASS / READY FOR ARCHIVED-VINTAGE PRODUCTION" if ACQUISITION_PASS else "REVIEW REQUIRED"}`

The next stage may use the validated local GeoTIFFs for the frozen V5 blockwise
feature production over LF2019L (2021 fires) and LF2020 (2022 fires).
"""

REPORT_PATH = (
    OUTPUT_ROOT /
    "LANDFIRE_ARCHIVE_ACQUISITION_REPORT_V7.md"
)

REPORT_PATH.write_text(
    report,
    encoding="utf-8",
)

print(
    report
)

print()
print(
    "Saved:",
    REPORT_PATH,
)

## 16. Output inventory

In [ ]:
print("=" * 100)
print("04D1G V5 OUTPUTS")
print("=" * 100)

for p in sorted(
    OUTPUT_ROOT.glob(
        "*"
    )
):
    if p.is_file():
        print(
            f"{p.name:<95}"
            f"{p.stat().st_size/(1024**2):>12.3f} MB"
        )

print()
print(
    "Archive files:",
    len(
        list(
            ARCHIVE_ROOT.rglob(
                "*.zip"
            )
        )
    ),
)

print(
    "Extracted GeoTIFF files:",
    len(
        list(
            EXTRACT_ROOT.rglob(
                "*.tif"
            )
        )
        + list(
            EXTRACT_ROOT.rglob(
                "*.tiff"
            )
        )
    ),
)

print()
print(
    "DONE:",
    NOTEBOOK_BUILD_ID,
)